# Tema 7 · Laboratorio — Optimizadores y learning rate

**Aprendizaje Profundo · CUNEF Universidad**

En este laboratorio entrenamos **el mismo MLP** sobre Fashion-MNIST cambiando solo el **optimizador** y el **learning rate**, y comparamos las **curvas de pérdida**. Es la versión "de verdad" de la práctica interactiva del tema.

Objetivos:
1. Ver cómo el optimizador (SGD, Momentum, Adam) cambia la velocidad y estabilidad del entrenamiento.
2. Ver cómo un learning rate demasiado grande **diverge** y uno demasiado pequeño va **lentísimo**.

> Ejecuta las celdas en orden. En Colab no necesitas instalar nada.

## 1 · Datos: Fashion-MNIST

70 000 imágenes de 28×28 de ropa en 10 clases. Normalizamos los píxeles a `[0, 1]` (un preprocesado que, por sí solo, ya ayuda a entrenar).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print('train:', x_train.shape, ' test:', x_test.shape)
class_names = ['Camiseta','Pantalón','Jersey','Vestido','Abrigo',
               'Sandalia','Camisa','Zapatilla','Bolso','Botín']

## 2 · Siempre el mismo modelo

Para que la comparación sea justa, definimos una función que construye **exactamente el mismo MLP** cada vez (misma arquitectura y misma semilla de inicialización). Lo único que cambiará entre experimentos es el **optimizador**.

In [ ]:
def build_mlp():
    """Un MLP pequeño y reproducible: 784 -> 128 -> 64 -> 10."""
    init = keras.initializers.GlorotUniform(seed=0)
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu', kernel_initializer=init),
        keras.layers.Dense(64, activation='relu', kernel_initializer=init),
        keras.layers.Dense(10, activation='softmax', kernel_initializer=init),
    ])
    return model

build_mlp().summary()

## 3 · Experimento A — tres optimizadores, mismo learning rate

Entrenamos con `SGD`, `SGD(momentum=0.9)` y `Adam`, todos con el **mismo** learning rate, durante unas pocas épocas. Guardamos el `history` de cada uno para comparar las curvas.

In [ ]:
EPOCHS = 12
BATCH = 128
LR = 0.01

optimizers = {
    'SGD':            keras.optimizers.SGD(learning_rate=LR),
    'SGD + momentum': keras.optimizers.SGD(learning_rate=LR, momentum=0.9),
    'Adam':           keras.optimizers.Adam(learning_rate=LR),
}

histories = {}
for name, opt in optimizers.items():
    print(f'\n=== Entrenando con {name} (lr={LR}) ===')
    model = build_mlp()
    model.compile(optimizer=opt,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    h = model.fit(x_train, y_train,
                  validation_split=0.1,
                  epochs=EPOCHS, batch_size=BATCH, verbose=2)
    histories[name] = h.history

In [ ]:
# Superponemos las curvas de pérdida (train) y de accuracy (validación)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in histories.items():
    ax1.plot(h['loss'], marker='o', label=name)
    ax2.plot(h['val_accuracy'], marker='o', label=name)
ax1.set_title('Pérdida en entrenamiento'); ax1.set_xlabel('época'); ax1.set_ylabel('loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Accuracy en validación'); ax2.set_xlabel('época'); ax2.set_ylabel('acc'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Para observar:** con el mismo learning rate, ¿qué optimizador baja antes la pérdida? Normalmente **Adam** y **momentum** superan al SGD puro, que va más lento. Este es el motivo por el que Adam es un punto de partida tan habitual.

## 4 · Experimento B — el learning rate lo cambia todo

Ahora fijamos el optimizador (**SGD**) y barremos varios learning rates. Veremos tres regímenes: **demasiado pequeño** (lento), **justo** (baja bien) y **demasiado grande** (rebota o diverge → la pérdida se dispara / `NaN`).

In [ ]:
lrs = [0.001, 0.01, 0.1, 1.0]
lr_hist = {}
for lr in lrs:
    print(f'\n=== SGD con learning_rate = {lr} ===')
    model = build_mlp()
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    h = model.fit(x_train, y_train,
                  validation_split=0.1,
                  epochs=EPOCHS, batch_size=BATCH, verbose=2)
    lr_hist[lr] = h.history

In [ ]:
plt.figure(figsize=(8, 5))
for lr, h in lr_hist.items():
    # recortamos valores enormes/NaN para que la gráfica siga siendo legible
    loss = np.array(h['loss'], dtype='float32')
    loss = np.nan_to_num(loss, nan=np.nan, posinf=np.nan)
    plt.plot(loss, marker='o', label=f'lr = {lr}')
plt.title('SGD · pérdida en entrenamiento según el learning rate')
plt.xlabel('época'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('Pérdida final por learning rate:')
for lr, h in lr_hist.items():
    print(f'  lr={lr:<6}: loss_final = {h["loss"][-1]:.4f}')

**Para observar:** con `lr=0.001` la pérdida baja despacio; con `lr=0.01`–`0.1` baja bien; con `lr=1.0` es probable que **diverja** (loss enorme o `NaN`). Es exactamente lo que pasaba en la práctica interactiva cuando subías demasiado el learning rate.

## 5 · Tus retos

1. **Batch size.** Vuelve al experimento A y prueba `BATCH = 16` y `BATCH = 512`. ¿Cómo cambia la suavidad de la curva y el tiempo por época?
2. **Adam aguanta más.** En el experimento B, cambia el optimizador a `Adam`. ¿Diverge también con `lr=1.0`, o soporta learning rates más altos que el SGD?
3. **Scheduling.** Añade un `keras.callbacks.LearningRateScheduler` (o `ReduceLROnPlateau`) para reducir el LR con las épocas. ¿Mejora la pérdida final?

Cuando termines, vuelve a la [práctica interactiva](../../practica-t7.html) y comprueba que tu intuición coincide con lo que has medido aquí.